<a href="https://colab.research.google.com/github/Terenceisstudying/SIT-UofG-QC-Assignment/blob/main/BB84-Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 80.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=ec47d690f974aa113ed5b5dfc8978181da16af1c9a720aebd4272407e7e5c3b5
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# Using the Alice and Bob example in the lecture.
# Alice creates random bits and random basis.
# Alice encodes each bit as a qubit.
# Alice send qubits to Bob.
# Bob chooses random basis and measures.
# Alice and Bob compare basis.
# Bob send bases to Alice
# Alice send only matching positions basis to Bob.

In [3]:
# Alice creates random Quantum bit
# To generate genuinely random numbers, we need a physical process that is random.
# For example: construct |+> and measure it.

def random_quantum_bit():
    qc = QuantumCircuit(1,1)
    qc.h(0)
    qc.measure(0,0)

    backend = BasicSimulator()
    compiled_circuit = transpile(qc, backend)
    job = backend.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    return int(list(counts.keys())[0])

# List of qubits
def random_quantum_bits(n):
    return [random_quantum_bit() for _ in range(n)]

In [4]:
# Alice encodes each bit
# bit|basis   |state
# 0  |standard| |0>
# 1  |standard| |1>
# 0  |diagonal| |+>
# 1  |diagonal| |->
def alice_encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)

    if basis == 1:
        qc.h(0)

    return qc

In [5]:
# Bob measures qubit

def bob_measure(prepared_qubit, basis):
    qc = prepared_qubit.copy()

    # if measuring in in diagonal basis (1), apply H
    if basis == 1:
        qc.h(0)
    # if measuring in standard basis (0), just do it
    qc.measure(0, 0)

    backend = BasicSimulator()
    compiled = transpile(qc, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)

    return int(list(counts.keys())[0])

In [6]:
# Converting of the basis to 's' standard, 'd' diagonal to follow the lecture's convention
def basis_label(basis):
      if basis == 0:
          return "s"
      return "d"
# Converting of the qubit label to follow the lecture's convention
def qubit_label(bit, basis):
      if basis == 0:
          return str(bit)
      if bit == 0:
          return "+"
      return "-"
# Converting of Bob's bit that does not match to follow the lecture's convention
def bob_display_bit(i):
      if alice_basis[i] == bob_basis[i]:
          return str(bob_bits[i])
      return "?"

In [7]:
n = 20

# Alice
alice_bits = random_quantum_bits(n)
alice_basis = random_quantum_bits(n)

qubits = []
for i in range(n):
    qubits.append(alice_encode_qubit(alice_bits[i], alice_basis[i]))

# Bob
bob_basis = random_quantum_bits(n)

bob_bits = []
for i in range(n):
    bob_bits.append(bob_measure(qubits[i], bob_basis[i]))

# Alice and Bob publicly basis comparison
matching_positions = []
for i in range(n):
    if alice_basis[i] == bob_basis[i]:
        matching_positions.append(i)

# Alice and Bob keep only the bits where their basis matched
alice_key = [alice_bits[i] for i in matching_positions]
bob_key = [bob_bits[i] for i in matching_positions]

# Print a table similar to Lecture 3b page 17.
cell_width = 4

print("Index:   ", end="")
for i in range(n):
    print(f"{i:<{cell_width}}", end="")
print()

print("A bit:   ", end="")
for bit in alice_bits:
    print(f"{bit:<{cell_width}}", end="")
print()

print("A basis: ", end="")
for basis in alice_basis:
    print(f"{basis_label(basis):<{cell_width}}", end="")
print()

print("qubit:   ", end="")
for i in range(n):
    print(f"{qubit_label(alice_bits[i], alice_basis[i]):<{cell_width}}", end="")
print()

print("B basis: ", end="")
for basis in bob_basis:
    print(f"{basis_label(basis):<{cell_width}}", end="")
print()

print("B bit:   ", end="")
for i in range(n):
    print(f"{bob_display_bit(i):<{cell_width}}", end="")
print()

print()
print("Matching positions:", matching_positions)
print("Alice key:", alice_key)
print("Bob key:  ", bob_key)
print("Keys match:", alice_key == bob_key)

Index:   0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  
A bit:   1   1   0   0   0   1   0   1   0   1   1   0   1   1   1   0   1   1   0   1   
A basis: d   d   s   d   d   d   d   s   d   s   d   s   d   s   d   d   d   s   d   d   
qubit:   -   -   0   +   +   -   +   1   +   1   -   0   -   1   -   +   -   1   +   -   
B basis: s   d   s   s   s   d   d   s   s   s   d   s   s   d   s   s   d   d   s   d   
B bit:   ?   1   0   ?   ?   1   0   1   ?   1   1   0   ?   ?   ?   ?   1   ?   ?   1   

Matching positions: [1, 2, 5, 6, 7, 9, 10, 11, 16, 19]
Alice key: [1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
Bob key:   [1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
Keys match: True
